# Estudo passo a passo do pipeline do TCC

**Previsão de Movimentos de Ativos Financeiros com Machine Learning**

Este notebook não substitui os scripts. Ele **usa as mesmas funções** deles, uma etapa por vez, para você enxergar cada DataFrame e entender cada decisão. Rode as células de cima para baixo com `Shift+Enter`.

> Antes, rode `python atualizar.py` pelo menos uma vez. O notebook lê os dados e os resultados que ele gerou.

Roteiro:
1. Configuração do experimento
2. Dados brutos
3. Indicadores e alvo (a parte mais importante)
4. Walk-forward e os modelos
5. Resultados

In [ ]:
import importlib

import numpy as np
import pandas as pd
from IPython.display import Image, display

import config

# os nomes dos scripts começam com número, por isso o import é feito assim
prep = importlib.import_module("02_preparar_variaveis")
modelar = importlib.import_module("03_modelar")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Configuração do experimento (`config.py`)

O `config.py` não executa nada: ele guarda **todas as decisões** do experimento num lugar só. O que o capítulo de metodologia disser precisa bater com este arquivo.

In [ ]:
pd.DataFrame(
    [(t, n, "ETF") for t, n in config.ETFS.items()]
    + [(t, n, "Ação") for t, n in config.ACOES.items()]
    + [(t, n, "FII") for t, n in config.FIIS.items()],
    columns=["ticker", "nome", "tipo"],
)

In [ ]:
pd.Series({
    "Início do download (aquecimento dos indicadores)": config.DATA_INICIO_DOWNLOAD,
    "Início da amostra analisada": config.DATA_INICIO,
    "Fim da amostra": config.DATA_FIM,
    "Início do período reservado": config.INICIO_RESERVADO,
    "Horizonte da previsão (pregões)": config.HORIZONTE,
    "Treino inicial (pregões)": config.TREINO_INICIAL,
    "Tamanho de cada teste (pregões)": config.JANELA_TESTE,
    "Modelos": ", ".join(config.MODELOS),
    "Variáveis": ", ".join(config.VARIAVEIS_TECNICAS),
}, name="valor").to_frame()

**Pergunta de banca:** *"Por que você baixou dados desde 2017 se a análise começa em 2018?"*

Alguns indicadores só existem depois de acumular histórico. O CCI precisa de 20 dias, o TRIX de cerca de 60 (três médias de 20) e o SMI de algumas dezenas. O ano extra serve de aquecimento: em 01/01/2018 todos os indicadores já têm valor.

## 2. Dados brutos (`01_baixar_dados.py`)

O script 01 baixa os 10 ativos, o VIX e o Ibovespa **em paralelo** com `concurrent.futures.ThreadPoolExecutor`, que foi a indicação do professor. Enquanto um download espera a resposta do Yahoo, outro já começa. Como esse tipo de tarefa é limitado pela rede e não pela CPU, threads funcionam bem.

Primeiro, veja o que foi baixado:

In [ ]:
pd.read_csv(config.DADOS_BRUTOS / "relatorio_download.csv")

In [ ]:
bova = prep.ler_precos("BOVA11")
print(bova.shape)          # (linhas = pregões, colunas)
bova.head()

In [ ]:
bova.describe().round(2)

In [ ]:
bova["fechamento"].plot(title="BOVA11 - fechamento ajustado", figsize=(10, 3));

**O que olhar:**
- Cada linha é um pregão. As colunas são abertura, máxima, mínima, fechamento e volume (OHLCV).
- O preço é **ajustado** (`auto_adjust=True`), ou seja, corrigido por dividendos e desdobramentos. Sem isso, um desdobramento pareceria uma queda de 50%.
- O CDI vem do Banco Central e é usado só no backtest, como o rendimento de "ficar no caixa":

In [ ]:
cdi = pd.read_csv(config.DADOS_BRUTOS / "cdi.csv", parse_dates=["data"])
cdi.tail()

## 3. Indicadores e alvo (`02_preparar_variaveis.py`)

### 3.1 Limpeza
A função `tratar` remove datas duplicadas e preços zerados, corrige máxima e mínima inconsistentes e descarta saltos acima de 50% num dia, que quase sempre são erro de dado.

In [ ]:
px = prep.tratar(bova)
print(len(bova), "->", len(px), "pregões após a limpeza")

### 3.2 Os 10 indicadores técnicos
`calcular_variaveis` recebe os preços e a variação do VIX e devolve uma linha por dia com todos os indicadores. Abra o `02_preparar_variaveis.py` ao lado e compare cada função com a definição do indicador.

In [ ]:
vix = prep.alinhar_vix(prep.variacao_vix(), px.index)
ind = prep.calcular_variaveis(px, vix)
ind[["preco"] + config.VARIAVEIS_TECNICAS].loc[config.DATA_INICIO:].head(10).round(3)

In [ ]:
ind[config.VARIAVEIS_TECNICAS].loc[config.DATA_INICIO:].describe().round(2)

| Indicador | O que mede | Faixa típica |
|---|---|---|
| `vix_var` | variação do "índice do medo" americano no dia anterior | próximo de 0 |
| `cci` | distância do preço à sua média de 20 dias | -200 a +200 |
| `macd` | diferença entre médias rápida (12) e lenta (26), em % | perto de 0 |
| `williams_r` | onde o fechamento está entre a máxima e a mínima de 14 dias | -100 a 0 |
| `stoch_smi` | versão suavizada do estocástico (13, 2, 25) | -100 a +100 |
| `trix` | variação de uma média exponencial tripla, em % | perto de 0 |
| `bollinger_pctb` | posição dentro das Bandas de Bollinger | 0 a 1 (às vezes fora) |
| `cmo` | força das altas contra as baixas em 14 dias | -100 a +100 |
| `dpo` | preço sem a tendência (ciclos) | perto de 0 |
| `roc` | retorno do dia | perto de 0 |

### 3.3 O alvo: o que o modelo tenta prever
`alvo = 1` se o preço **de amanhã** for maior que o de hoje, e `0` caso contrário. O `ret_futuro` existe **só** para construir o alvo e calcular o backtest; ele **nunca** entra como variável do modelo.

In [ ]:
dados = pd.read_parquet(config.DADOS_PROCESSADOS / "dataset.parquet")
b = dados[dados["ativo"] == "BOVA11"].set_index("data")
b[["preco", "roc", "ret_futuro", "alvo"]].head(8)

Repare que o `ret_futuro` de um dia é igual ao `roc` **do dia seguinte**. É o mesmo número visto de dois momentos diferentes: hoje ele é desconhecido (é o que queremos prever) e amanhã ele vira informação conhecida.

### 3.4 A prova de que os indicadores não olham o futuro
Este é o ponto que mais derruba trabalhos de previsão financeira, e a banca de Computação costuma perguntar. O teste: calcule os indicadores **cortando a série num dia D**, como se o futuro não existisse, e compare com o cálculo feito usando a série inteira. Se o dia D der igual nos dois, nenhum indicador usou dados posteriores a D.

In [ ]:
dia = ind.loc[config.DATA_INICIO:].index[300]
ind_cortado = prep.calcular_variaveis(px.loc[:dia], vix.loc[:dia])
iguais = np.allclose(ind.loc[dia, config.VARIAVEIS_TECNICAS].astype(float),
                     ind_cortado.loc[dia, config.VARIAVEIS_TECNICAS].astype(float), equal_nan=True)
print(f"Dia testado: {dia.date()} -> indicadores iguais com e sem o futuro? {iguais}")

**Pergunta de banca:** *"Como você garante que não houve vazamento de dados?"*

Três cuidados:
1. todos os indicadores são calculados só com o passado, como o teste acima mostra;
2. o VIX entra com um dia de atraso, porque o mercado americano fecha depois da B3;
3. o DPO usa a versão causal, sem deslocar a média para o futuro.

In [ ]:
print("Observações por ativo:")
display(dados.groupby(["tipo", "ativo"]).size().rename("dias").to_frame())
print("Proporção de dias de alta (alvo = 1):", round(dados["alvo"].mean(), 3))

A proporção de altas fica perto de 50%. Por isso **acertar 50% é o que se consegue no chute**, e a pergunta real é se os modelos fazem melhor do que referências simples.

## 4. Walk-forward e os modelos (`03_modelar.py`)

### 4.1 As janelas
No walk-forward o modelo **estuda o passado e faz a prova do futuro**, avançando no tempo. Primeiro treina em cerca de 2 anos e testa nos 6 meses seguintes. Depois incorpora esses 6 meses ao treino e testa os próximos 6, e assim por diante. Os dados nunca são embaralhados.

In [ ]:
b = dados[dados["ativo"] == "BOVA11"].sort_values("data").reset_index(drop=True)
janelas = []
for fold, tr, te in modelar.walk_forward(len(b)):
    janelas.append({"fold": fold,
                    "treino": f"{b.loc[tr[0], 'data'].date()} a {b.loc[tr[-1], 'data'].date()}",
                    "dias treino": len(tr),
                    "teste": f"{b.loc[te[0], 'data'].date()} a {b.loc[te[-1], 'data'].date()}",
                    "dias teste": len(te)})
pd.DataFrame(janelas)

**Pergunta de banca:** *"Por que não usou validação cruzada comum (k-fold)?"*

Porque o k-fold embaralha os dias. O modelo treinaria com 2024 e seria testado em 2020, ou seja, aprenderia com o futuro. Em séries temporais isso infla a acurácia artificialmente. O walk-forward respeita a ordem do tempo, exatamente como aconteceria na vida real.

### 4.2 Um fold "na mão"
O script faz isto automaticamente para todos os folds e ativos. Aqui está só o primeiro fold do BOVA11, para você ver cada passo.

In [ ]:
fold, tr, te = next(modelar.walk_forward(len(b)))
X = b[config.VARIAVEIS_TECNICAS].values
y = b["alvo"].astype(int).values

logistica = modelar.novo_modelo("logistica").fit(X[tr], y[tr])
floresta = modelar.novo_modelo("random_forest").fit(X[tr], y[tr])

resultado = b.loc[te, ["data", "alvo"]].copy()
resultado["prob_logistica"] = logistica.predict_proba(X[te])[:, 1]
resultado["prob_rf"] = floresta.predict_proba(X[te])[:, 1]
resultado["pred_logistica"] = (resultado["prob_logistica"] >= 0.5).astype(int)
resultado["pred_rf"] = (resultado["prob_rf"] >= 0.5).astype(int)
resultado.head(10).round(3)

In [ ]:
print("Acurácia neste fold:")
print("  Regressão Logística:", round((resultado["pred_logistica"] == resultado["alvo"]).mean(), 3))
print("  Random Forest:      ", round((resultado["pred_rf"] == resultado["alvo"]).mean(), 3))
print("  Sempre alta:        ", round((resultado["alvo"] == 1).mean(), 3))

**Como cada modelo decide:**
- A **Regressão Logística** dá um peso para cada indicador, soma tudo e transforma o resultado numa probabilidade. Os pesos abaixo mostram para que lado cada indicador "empurra" a previsão. Como as variáveis foram padronizadas, dá para comparar os pesos entre si.
- A **Random Forest** monta 300 árvores de perguntas do tipo "o CCI está acima de X?" e faz uma votação entre elas. A importância de cada variável mostra quais indicadores as árvores mais usaram para separar altas de baixas.

In [ ]:
pd.DataFrame({
    "peso_logistica": logistica[-1].coef_[0],
    "importancia_rf": floresta.feature_importances_,
}, index=config.VARIAVEIS_TECNICAS).sort_values("importancia_rf", ascending=False).round(3)

## 5. Resultados (`resultados/`)

Daqui em diante entram os resultados de **todos os folds de todos os ativos**, gerados pelo script 03. Isto é a matéria-prima do capítulo de Resultados e Discussão.

In [ ]:
resumo = pd.read_csv(config.RESULTADOS / "metricas_resumo.csv")
resumo[resumo["segmento"].isin(["geral", "periodo"])][
    ["segmento", "valor", "modelo", "n", "acuracia", "auc", "f1", "p_valor_vs_sempre_alta"]
].round(3)

**Como ler:**
- **acuracia:** fração de dias em que o modelo acertou a direção.
- **auc:** capacidade de separar altas de baixas pela probabilidade. 0,5 é o mesmo que o acaso.
- **p_valor_vs_sempre_alta:** chance de o modelo superar "sempre alta" só por sorte. Abaixo de 0,05, a vantagem é estatisticamente significativa.
- **período reservado:** o último ano, que funciona como prova final. Se o resultado se mantém nele, a conclusão é mais confiável.

In [ ]:
comp = pd.read_csv(config.RESULTADOS / "comparacao_modelos.csv")
comp[comp["segmento"].isin(["geral", "periodo", "ano"])][
    ["segmento", "valor", "acuracia_a", "acuracia_b", "so_a_acerta", "so_b_acerta", "p_valor_mcnemar"]
].rename(columns={"acuracia_a": "acc_logistica", "acuracia_b": "acc_rf",
                  "so_a_acerta": "so_logistica_acerta", "so_b_acerta": "so_rf_acerta"}).round(4)

**Teste de McNemar: a pergunta central do TCC.** O teste olha só os dias em que **um modelo acertou e o outro errou**. Se a Random Forest fosse realmente melhor, ela "ganharia" a maioria desses dias. Um p-valor abaixo de 0,05 indica que a diferença entre os modelos não se explica por acaso.

A linha "ano" responde à segunda parte da pergunta: **a vantagem se mantém ao longo do tempo?**

In [ ]:
pd.read_csv(config.RESULTADOS / "estabilidade.csv")

In [ ]:
por_ano = pd.read_csv(config.RESULTADOS / "metricas_por_ano.csv")
tab = por_ano.pivot(index="ano", columns="modelo", values="acuracia")
tab.round(3)

In [ ]:
for fig in sorted(config.FIGURAS.glob("*.png")):
    print(fig.name)
    display(Image(filename=str(fig), width=700))

---
**Próximos passos:** anote as dúvidas e os números que chamarem atenção. É a partir deles que vamos escrever o capítulo de Resultados e Discussão.